In [1]:
import os
import time
import warnings
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")


# =========================================================
# SETTINGS YOU CAN EDIT
# =========================================================
TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENT_NAME = "Spotify_Churn_Logistic_Regression"
REGISTERED_MODEL_NAME = "spotify_churn_logreg_best"

# Pick the C value for the model you decided to keep from notebook 05.
# Example: 0.1, 1.0, 10.0, etc.
CHOSEN_C = 0.1

# If you want to tag this registered version as the current best:
SET_CHAMPION_ALIAS = True

# File path based on your uploaded code
FILE_NAME = "../data/processed/spotify_user_behavior_cleaned.parquet"
TARGET_COLUMN = "churn"


# =========================================================
# HELPER FUNCTIONS
# =========================================================
def build_pipeline(X_frame, selected_features, c_val):
    X_subset = X_frame[selected_features]

    numeric_features = X_subset.select_dtypes(
        include=["int64", "int32", "float64", "float32"]
    ).columns.tolist()

    categorical_features = X_subset.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )

    model_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            C=c_val,
            max_iter=3000,
            random_state=42,
            class_weight="balanced",
            solver="liblinear",
            penalty="l2"
        ))
    ])

    return model_pipeline


def backward_selection(X_train, y_train, X_val, y_val, initial_features, c_val, min_features=3):
    selected_features = initial_features.copy()

    baseline_pipeline = build_pipeline(X_train, selected_features, c_val)
    baseline_pipeline.fit(X_train[selected_features], y_train)
    baseline_preds = baseline_pipeline.predict(X_val[selected_features])
    best_score = f1_score(y_val, baseline_preds, zero_division=0)

    improved = True

    while improved and len(selected_features) > min_features:
        improved = False
        feature_to_remove = None
        candidate_best_score = best_score

        for feature in selected_features:
            trial_features = [f for f in selected_features if f != feature]

            trial_pipeline = build_pipeline(X_train, trial_features, c_val)
            trial_pipeline.fit(X_train[trial_features], y_train)
            trial_preds = trial_pipeline.predict(X_val[trial_features])
            trial_score = f1_score(y_val, trial_preds, zero_division=0)

            if trial_score >= candidate_best_score:
                candidate_best_score = trial_score
                feature_to_remove = feature

        if feature_to_remove is not None:
            selected_features.remove(feature_to_remove)
            best_score = candidate_best_score
            improved = True

    return selected_features, best_score


def save_roc_curve(y_test, y_prob, output_file):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_value = roc_auc_score(y_test, y_prob)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {auc_value:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve - Registered Best Model")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(output_file)
    plt.close()


# =========================================================
# MLFLOW SETUP
# =========================================================
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow Tracking URI:", TRACKING_URI)
print("Experiment:", EXPERIMENT_NAME)
print("Registered Model Name:", REGISTERED_MODEL_NAME)
print("Chosen C:", CHOSEN_C)
print()


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_parquet(FILE_NAME)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())
print()

if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' not found in dataset.")

df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")
df["days_since_signup_start"] = (
    df["signup_date"] - df["signup_date"].min()
).dt.days

drop_columns = ["user_id", "signup_date"]

df = df.dropna(subset=[TARGET_COLUMN])

X = df.drop(columns=drop_columns + [TARGET_COLUMN], errors="ignore")
y = df[TARGET_COLUMN].astype(int)

print("Target distribution:")
print(y.value_counts())
print()


# =========================================================
# SPLITS
# =========================================================
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# validation split for backward selection
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full
)

initial_features = X.columns.tolist()


# =========================================================
# TRAIN THE CHOSEN MODEL
# =========================================================
with mlflow.start_run(run_name=f"logreg_best_registered_C_{CHOSEN_C}") as run:
    # Backward selection timing
    selection_start = time.perf_counter()
    selected_features, selection_val_f1 = backward_selection(
        X_train,
        y_train,
        X_val,
        y_val,
        initial_features,
        CHOSEN_C,
        min_features=3
    )
    selection_end = time.perf_counter()

    removed_features = [f for f in initial_features if f not in selected_features]

    # Train final model on full training split
    final_pipeline = build_pipeline(X_train_full, selected_features, CHOSEN_C)

    train_start = time.perf_counter()
    final_pipeline.fit(X_train_full[selected_features], y_train_full)
    train_end = time.perf_counter()

    # Predict
    predict_start = time.perf_counter()
    y_pred = final_pipeline.predict(X_test[selected_features])
    y_prob = final_pipeline.predict_proba(X_test[selected_features])[:, 1]
    predict_end = time.perf_counter()

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)

    selection_time = selection_end - selection_start
    train_time = train_end - train_start
    predict_time = predict_end - predict_start
    total_runtime = selection_time + train_time + predict_time

    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred)

    # Print results
    print("=" * 60)
    print("REGISTERING CHOSEN MODEL")
    print("Selected features:")
    print(selected_features)
    print()
    print("Removed features:")
    print(removed_features)
    print()
    print("Selection validation F1:", selection_val_f1)
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    print("ROC AUC:", roc_auc)
    print("Selection Time:", selection_time)
    print("Train Time:", train_time)
    print("Predict Time:", predict_time)
    print("Total Runtime:", total_runtime)
    print("Confusion Matrix:")
    print(cm)
    print("Classification Report:")
    print(report)

    # Log params
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("selection_method", "backward")
    mlflow.log_param("dataset_file", FILE_NAME)
    mlflow.log_param("target_column", TARGET_COLUMN)
    mlflow.log_param("C", CHOSEN_C)
    mlflow.log_param("max_iter", 3000)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("solver", "liblinear")
    mlflow.log_param("penalty", "l2")
    mlflow.log_param("train_rows", len(X_train_full))
    mlflow.log_param("test_rows", len(X_test))
    mlflow.log_param("selected_feature_count", len(selected_features))
    mlflow.log_param("removed_feature_count", len(removed_features))
    mlflow.log_param("selected_features", ",".join(selected_features))
    mlflow.log_param("removed_features", ",".join(removed_features))
    mlflow.log_param("chosen_reason", "shortest_processing_time")

    # Log metrics
    mlflow.log_metric("selection_validation_f1", selection_val_f1)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("selection_time_seconds", selection_time)
    mlflow.log_metric("train_time_seconds", train_time)
    mlflow.log_metric("predict_time_seconds", predict_time)
    mlflow.log_metric("total_runtime_seconds", total_runtime)

    # Save selected features file
    safe_c_val = str(CHOSEN_C).replace(".", "_")
    selected_file = f"selected_registered_C_{safe_c_val}.csv"
    pd.DataFrame({"selected_feature": selected_features}).to_csv(selected_file, index=False)
    mlflow.log_artifact(selected_file)

    # Save ROC curve
    roc_file = f"roc_registered_C_{safe_c_val}.png"
    save_roc_curve(y_test, y_prob, roc_file)
    mlflow.log_artifact(roc_file)

    # Register the model
    model_info = mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path="registered_model_artifact",
        registered_model_name=REGISTERED_MODEL_NAME
    )

    print()
    print("Run ID:", run.info.run_id)
    print("Model logged and registration requested.")
    print("Registered model name:", REGISTERED_MODEL_NAME)


# =========================================================
# OPTIONAL: SET ALIAS 'champion' TO THE LATEST VERSION
# =========================================================
if SET_CHAMPION_ALIAS:
    client = MlflowClient(tracking_uri=TRACKING_URI)

    versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
    latest_version_number = max(int(v.version) for v in versions)

    client.set_registered_model_alias(
        name=REGISTERED_MODEL_NAME,
        alias="champion",
        version=latest_version_number
    )

    print()
    print(f"Latest registered version: {latest_version_number}")
    print("Alias 'champion' assigned successfully.")


print()
print("=" * 60)
print("DONE")
print("Open MLflow UI at: http://127.0.0.1:5000")
print(f"Check the Models tab for: {REGISTERED_MODEL_NAME}")
print("If this is your first registration, it should appear as Version 1.")


MLflow Tracking URI: http://127.0.0.1:5000
Experiment: Spotify_Churn_Logistic_Regression
Registered Model Name: spotify_churn_logreg_best
Chosen C: 1.0

Dataset loaded successfully.
Shape: (50000, 18)
Columns:
['user_id', 'country', 'age', 'signup_date', 'subscription_type', 'churn', 'months_inactive', 'inactive_3_months_flag', 'ad_interaction', 'ad_conversion_to_subscription', 'music_suggestion_rating_1_to_5', 'avg_listening_hours_per_week', 'favorite_genre', 'most_liked_feature', 'desired_future_feature', 'primary_device', 'playlists_created', 'avg_skips_per_day']

Target distribution:
churn
0    42109
1     7891
Name: count, dtype: int64

REGISTERING CHOSEN MODEL
Selected features:
['country', 'age', 'inactive_3_months_flag']

Removed features:
['subscription_type', 'months_inactive', 'ad_interaction', 'ad_conversion_to_subscription', 'music_suggestion_rating_1_to_5', 'avg_listening_hours_per_week', 'favorite_genre', 'most_liked_feature', 'desired_future_feature', 'primary_device', 

2026/04/16 00:26:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 00:26:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'spotify_churn_logreg_best' already exists. Creating a new version of this model...
2026/04/16 00:26:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: spotify_churn_logreg_best, version 2
Created version '2' of model 'spotify_churn_logreg_best'.



Run ID: 122d6b0ddee34d53821e3d0808234fb7
Model logged and registration requested.
Registered model name: spotify_churn_logreg_best
🏃 View run logreg_best_registered_C_1.0 at: http://127.0.0.1:5000/#/experiments/1/runs/122d6b0ddee34d53821e3d0808234fb7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

Latest registered version: 2
Alias 'champion' assigned successfully.

DONE
Open MLflow UI at: http://127.0.0.1:5000
Check the Models tab for: spotify_churn_logreg_best
If this is your first registration, it should appear as Version 1.
